In [1]:
import torch
print(f"GPUs: {torch.cuda.device_count()}")
# Should print: GPUs: 2

GPUs: 2


In [2]:
"""
============================================================
ETT (Electricity Transformer Temperature) Load Forecasting
LW-AdaDelta vs Baselines — Autoregressive Temporal Benchmark
============================================================
Single session — runs all 108 experiments in ~2 hours on T4x2.
Checkpoint saved after every experiment.

Speed optimizations vs original:
  • batch_size 32 → 64   (~1.5x faster)
  • num_workers 4 → 2    (Kaggle stable)
  • DataParallel added   (~1.8x from 2 GPUs)
  • Checkpointing added  (resume on interruption)
"""

import os
import math
import time
import pickle
import warnings
from collections import deque
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

CHECKPOINT_PATH = '/kaggle/working/ett_checkpoint.pkl'

# ============================================================
# LW-AdaDelta Optimizer
# ============================================================

class LWAdaDelta(torch.optim.Optimizer):
    """Local-Window AdaDelta with temporal smoothing"""

    def __init__(self, params, rho=0.9, eps=1e-6, k=2, tau=1.0):
        defaults = dict(rho=rho, eps=eps, k=k, tau=tau)
        super().__init__(params, defaults)
        self._weight_cache = {}

    def _weights(self, k, tau, device):
        key = (k, tau, device)
        if key not in self._weight_cache:
            w = torch.tensor(
                [math.exp(-i / tau) for i in range(k)], device=device
            )
            self._weight_cache[key] = w / w.sum()
        return self._weight_cache[key]

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            rho, eps, k, tau = group["rho"], group["eps"], group["k"], group["tau"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad  = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["Eg2"]  = torch.zeros_like(p)
                    state["Edx2"] = torch.zeros_like(p)
                    state["buf"]  = deque(maxlen=k)
                Eg2, Edx2, buf = state["Eg2"], state["Edx2"], state["buf"]
                Eg2.mul_(rho).addcmul_(grad, grad, value=1 - rho)
                rms_dx    = torch.sqrt(Edx2 + eps)
                rms_g     = torch.sqrt(Eg2  + eps)
                delta_raw = -(rms_dx / rms_g) * grad
                buf.append(delta_raw.detach())
                weights = self._weights(len(buf), tau, p.device)
                delta   = torch.zeros_like(p)
                for w, u in zip(weights, reversed(buf)):
                    delta.add_(u, alpha=w.item())
                p.add_(delta)
                Edx2.mul_(rho).addcmul_(delta, delta, value=1 - rho)


class Lion(torch.optim.Optimizer):
    """Lion optimizer"""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad    = p.grad
                state   = self.state[p]
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                exp_avg      = state['exp_avg']
                beta1, beta2 = group['betas']
                if group['weight_decay'] != 0:
                    p.mul_(1 - group['lr'] * group['weight_decay'])
                update = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                p.add_(torch.sign(update), alpha=-group['lr'])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)


# ============================================================
# Data Download & Loading
# ============================================================

ETT_URLS = {
    'ETTh1': ('https://raw.githubusercontent.com/zhouhaoyi/'
               'ETDataset/main/ETT-small/ETTh1.csv'),
    'ETTh2': ('https://raw.githubusercontent.com/zhouhaoyi/'
               'ETDataset/main/ETT-small/ETTh2.csv'),
}


def download_ett(data_dir='./ett_data'):
    """Download ETTh1 and ETTh2 — ~3 MB each, takes ~5 seconds."""
    import urllib.request
    os.makedirs(data_dir, exist_ok=True)
    paths = {}
    for name, url in ETT_URLS.items():
        path = os.path.join(data_dir, f'{name}.csv')
        if not os.path.exists(path):
            print(f"[INFO] Downloading {name} …")
            urllib.request.urlretrieve(url, path)
            print(f"[INFO] Saved → {path}")
        else:
            print(f"[INFO] {name} already exists")
        paths[name] = path
    return paths


def load_ett(csv_path):
    """Load ETT CSV. 7 features including OT (oil temperature)."""
    df        = pd.read_csv(csv_path).drop(columns=['date'])
    col_names = df.columns.tolist()
    data      = df.values.astype(np.float32)
    print(f"[INFO] Loaded {os.path.basename(csv_path)}: "
          f"shape={data.shape}, cols={col_names}")
    return data, col_names


# ============================================================
# Dataset
# ============================================================

class ETTDataset(Dataset):
    """Sliding window for autoregressive forecasting."""
    def __init__(self, data, seq_len=336, pred_len=24, stride=1):
        self.data     = data
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.indices  = list(range(
            0, len(data) - seq_len - pred_len + 1, stride
        ))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        s = self.indices[idx]
        x = self.data[s : s + self.seq_len]
        y = self.data[s + self.seq_len : s + self.seq_len + self.pred_len]
        return torch.tensor(x), torch.tensor(y)


def split_and_scale(data, train_ratio=0.7, val_ratio=0.1):
    """Chronological split — no shuffle to preserve time series integrity."""
    n    = len(data)
    n_tr = int(n * train_ratio)
    n_vl = int(n * val_ratio)

    tr_raw = data[:n_tr]
    vl_raw = data[n_tr : n_tr + n_vl]
    te_raw = data[n_tr + n_vl :]

    mean = tr_raw.mean(axis=0, keepdims=True)
    std  = tr_raw.std(axis=0,  keepdims=True) + 1e-8

    print(f"[INFO] Train={len(tr_raw)}, Val={len(vl_raw)}, "
          f"Test={len(te_raw)} timesteps")
    return (tr_raw-mean)/std, (vl_raw-mean)/std, (te_raw-mean)/std, \
           {'mean': mean, 'std': std}


def make_loaders(train, val, test, seq_len=336, pred_len=24,
                 batch_size=64):     # ← 64 (doubled from original 32)
    train_ds = ETTDataset(train, seq_len, pred_len)
    val_ds   = ETTDataset(val,   seq_len, pred_len)
    test_ds  = ETTDataset(test,  seq_len, pred_len)

    kw = dict(num_workers=2, pin_memory=True)   # ← num_workers=2
    train_loader = DataLoader(train_ds, batch_size, shuffle=True,  **kw)
    val_loader   = DataLoader(val_ds,   batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size, shuffle=False, **kw)

    print(f"[INFO] Loader sizes — Train:{len(train_ds)} "
          f"Val:{len(val_ds)} Test:{len(test_ds)}")
    return train_loader, val_loader, test_loader


# ============================================================
# Model: Dilated TCN
# ============================================================

class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, dilation=1):
        super().__init__()
        self.pad  = (kernel - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel,
                              padding=0, dilation=dilation, bias=False)

    def forward(self, x):
        return self.conv(F.pad(x, (self.pad, 0)))


class TCNBlock(nn.Module):
    def __init__(self, n_in, n_out, kernel=3, dilation=1, dropout=0.1):
        super().__init__()
        self.conv1 = CausalConv1d(n_in,  n_out, kernel, dilation)
        self.bn1   = nn.BatchNorm1d(n_out)
        self.drop1 = nn.Dropout(dropout)
        self.conv2 = CausalConv1d(n_out, n_out, kernel, dilation)
        self.bn2   = nn.BatchNorm1d(n_out)
        self.drop2 = nn.Dropout(dropout)
        self.ds    = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None

    def forward(self, x):
        out = self.drop1(F.relu(self.bn1(self.conv1(x))))
        out = self.drop2(F.relu(self.bn2(self.conv2(out))))
        return F.relu(out + (x if self.ds is None else self.ds(x)))


class TCNForecaster(nn.Module):
    """
    Dilated TCN for multivariate forecasting.
    Input : [B, seq_len, n_features]
    Output: [B, pred_len, n_features]
    """
    def __init__(self, n_features=7, seq_len=336, pred_len=24,
                 n_channels=64, n_layers=6, kernel=3, dropout=0.1):
        super().__init__()
        self.seq_len    = seq_len
        self.pred_len   = pred_len
        self.n_features = n_features

        layers, in_ch = [], n_features
        for i in range(n_layers):
            layers.append(TCNBlock(in_ch, n_channels,
                                   kernel, 2**i, dropout))
            in_ch = n_channels
        self.tcn        = nn.Sequential(*layers)
        self.projection = nn.Linear(n_channels * seq_len,
                                    pred_len * n_features)

    def forward(self, x):
        x   = x.permute(0, 2, 1)
        out = self.tcn(x)
        out = self.projection(out.reshape(out.size(0), -1))
        return out.view(out.size(0), self.pred_len, self.n_features)


# ============================================================
# Optimizer Factory
# ============================================================

def get_optimizer(name, params):
    if name == 'SGD':
        return torch.optim.SGD(params, lr=0.01,
                               momentum=0.9, weight_decay=1e-5)
    elif name == 'Adam':
        return torch.optim.Adam(params, lr=0.001, weight_decay=1e-5)
    elif name == 'AdamW':
        return torch.optim.AdamW(params, lr=0.001, weight_decay=1e-4)
    elif name == 'Lion':
        return Lion(params, lr=1e-4, weight_decay=1e-4)
    elif name == 'AdaDelta':
        return torch.optim.Adadelta(params, rho=0.9)
    elif name == 'LW-AdaDelta':
        return LWAdaDelta(params, rho=0.9, k=2, tau=1.0)
    else:
        raise ValueError(f"Unknown optimizer: {name}")


# ============================================================
# Training
# ============================================================

def train_epoch(model, optimizer, loader, device):
    model.train()
    total = 0.0
    for x_b, y_b in loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = F.l1_loss(model(x_b), y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    """MAE, RMSE, MAPE on OT (last feature)."""
    model.eval()
    preds_list, tgts_list = [], []
    for x_b, y_b in loader:
        preds_list.append(model(x_b.to(device)).cpu().numpy())
        tgts_list.append(y_b.numpy())

    p = np.concatenate(preds_list, axis=0)[:, :, -1].flatten()
    t = np.concatenate(tgts_list,  axis=0)[:, :, -1].flatten()

    mae  = np.mean(np.abs(p - t))
    rmse = np.sqrt(np.mean((p - t)**2))
    mape = np.mean(np.abs((p - t) / (np.abs(t) + 1e-8))) * 100
    return mae, rmse, mape


def train_with_early_stopping(model, optimizer, train_loader,
                               val_loader, device,
                               min_epochs=20, max_epochs=150,
                               patience=15):
    best_mae   = float('inf')
    best_state = None
    no_imp     = 0
    history    = {'train_loss': [], 'val_mae': [],
                  'val_rmse': [], 'val_mape': [], 'epoch_times': []}

    for epoch in range(max_epochs):
        t0      = time.time()
        tr_loss = train_epoch(model, optimizer, train_loader, device)
        v_mae, v_rmse, v_mape = evaluate(model, val_loader, device)
        elapsed = time.time() - t0

        history['train_loss'].append(tr_loss)
        history['val_mae'].append(v_mae)
        history['val_rmse'].append(v_rmse)
        history['val_mape'].append(v_mape)
        history['epoch_times'].append(elapsed)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}: Loss={tr_loss:.4f}  "
                  f"ValMAE={v_mae:.4f}  ValRMSE={v_rmse:.4f}  "
                  f"ValMAPE={v_mape:.2f}%  Time={elapsed:.1f}s")

        if v_mae < best_mae:
            best_mae   = v_mae
            best_state = deepcopy(model.state_dict())
            no_imp     = 0
        else:
            no_imp += 1

        if epoch >= min_epochs and no_imp >= patience:
            print(f"  Early stopping at epoch {epoch+1} "
                  f"(no improvement for {patience} epochs)")
            break

    if best_state:
        model.load_state_dict(best_state)

    history['best_val_mae'] = best_mae
    history['total_epochs'] = epoch + 1
    return history


# ============================================================
# Single Experiment
# ============================================================

def run_experiment(train_data, val_data, test_data,
                   seq_len, pred_len, n_features,
                   optimizer_name, seed, device):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    train_loader, val_loader, test_loader = make_loaders(
        train_data, val_data, test_data,
        seq_len=seq_len, pred_len=pred_len, batch_size=64
    )

    model = TCNForecaster(
        n_features=n_features, seq_len=seq_len, pred_len=pred_len,
        n_channels=64, n_layers=6, kernel=3, dropout=0.1
    ).to(device)

    # ── DataParallel for T4x2 ────────────────────────────────────
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    optimizer = get_optimizer(optimizer_name, model.parameters())

    history = train_with_early_stopping(
        model, optimizer, train_loader, val_loader, device,
        min_epochs=20, max_epochs=150, patience=15
    )

    test_mae, test_rmse, test_mape = evaluate(model, test_loader, device)

    print(f"  Test — MAE={test_mae:.4f}  "
          f"RMSE={test_rmse:.4f}  MAPE={test_mape:.2f}%")

    return {
        'history':    history,
        'test_mae':   test_mae,
        'test_rmse':  test_rmse,
        'test_mape':  test_mape,
        'total_time': sum(history['epoch_times'])
    }


# ============================================================
# Full Benchmark — checkpointed
# ============================================================

def run_full_benchmark(ett_paths, device='cuda', seeds=[0, 1, 2]):
    optimizers    = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    pred_horizons = [24, 48, 168]
    seq_len       = 336

    # Load checkpoint if it exists
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, 'rb') as f:
            results = pickle.load(f)
        done = sum(
            len(results[ds][h][o])
            for ds in ett_paths
            for h  in pred_horizons
            for o  in optimizers
        )
        print(f"[INFO] Checkpoint loaded — {done}/108 already done")
    else:
        results = {
            ds: {h: {opt: [] for opt in optimizers}
                 for h in pred_horizons}
            for ds in ett_paths
        }
        print("[INFO] No checkpoint — starting fresh")

    total = len(ett_paths) * len(pred_horizons) * len(optimizers) * len(seeds)
    done  = 0

    for ds_name, csv_path in ett_paths.items():
        raw_data, col_names = load_ett(csv_path)
        n_features          = raw_data.shape[1]
        train_d, val_d, test_d, _ = split_and_scale(raw_data)

        for pred_len in pred_horizons:
            print(f"\n{'#'*70}")
            print(f"# Dataset={ds_name}  Horizon={pred_len}h  "
                  f"Seq={seq_len}  Features={n_features}")
            print(f"{'#'*70}")

            for opt_name in optimizers:
                for seed in seeds:
                    done += 1

                    # Skip already-done experiments
                    if len(results[ds_name][pred_len][opt_name]) > seed:
                        print(f"  Skipping {ds_name} H={pred_len} "
                              f"{opt_name} seed={seed}")
                        continue

                    print(f"\n[Progress {done}/{total}] "
                          f"{ds_name} H={pred_len} {opt_name} seed={seed}")

                    result = run_experiment(
                        train_d, val_d, test_d,
                        seq_len=seq_len,
                        pred_len=pred_len,
                        n_features=n_features,
                        optimizer_name=opt_name,
                        seed=seed,
                        device=device
                    )
                    results[ds_name][pred_len][opt_name].append(result)

                    # Save checkpoint immediately
                    with open(CHECKPOINT_PATH, 'wb') as f:
                        pickle.dump(results, f)
                    print(f"  ✓ Checkpoint saved "
                          f"({ds_name} H={pred_len} {opt_name} seed={seed})")

    return results


# ============================================================
# Analysis
# ============================================================

def analyze_results(results):
    optimizers    = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    pred_horizons = sorted(list(next(iter(results.values())).keys()))
    all_metrics   = {}

    print("\n" + "="*80)
    print("ETT LOAD FORECASTING — COMPREHENSIVE RESULTS")
    print("="*80)

    for ds_name, ds_res in results.items():
        all_metrics[ds_name] = {}
        print(f"\n{'='*80}\nDATASET: {ds_name}\n{'='*80}")

        for pred_len in pred_horizons:
            all_metrics[ds_name][pred_len] = {}
            print(f"\n  Horizon = {pred_len}h")
            print(f"  {'Optimizer':<15} {'MAE':<20} {'RMSE':<20} "
                  f"{'MAPE':<18} {'Epochs'}")
            print(f"  {'-'*75}")

            for opt in optimizers:
                runs   = ds_res[pred_len][opt]
                maes   = [r['test_mae']  for r in runs]
                rmses  = [r['test_rmse'] for r in runs]
                mapes  = [r['test_mape'] for r in runs]
                epochs = [r['history']['total_epochs'] for r in runs]
                m = {
                    'mae_mean':  np.mean(maes),  'mae_std':  np.std(maes),
                    'rmse_mean': np.mean(rmses), 'rmse_std': np.std(rmses),
                    'mape_mean': np.mean(mapes), 'mape_std': np.std(mapes),
                    'ep_mean':   np.mean(epochs),
                }
                all_metrics[ds_name][pred_len][opt] = m
                print(f"  {opt:<15} "
                      f"{m['mae_mean']:.4f}±{m['mae_std']:.4f}    "
                      f"{m['rmse_mean']:.4f}±{m['rmse_std']:.4f}    "
                      f"{m['mape_mean']:.2f}±{m['mape_std']:.2f}%    "
                      f"{m['ep_mean']:.0f}")

            lw  = all_metrics[ds_name][pred_len]['LW-AdaDelta']
            ada = all_metrics[ds_name][pred_len]['AdaDelta']
            d_mae  = ada['mae_mean']  - lw['mae_mean']
            d_rmse = ada['rmse_mean'] - lw['rmse_mean']
            pct    = 100.0 * d_mae / (ada['mae_mean'] + 1e-8)

            print(f"\n  LW-AdaDelta vs AdaDelta:")
            print(f"    ΔMAE  = {d_mae:+.4f} ({pct:+.1f}%)  "
                  + ("✓ LW wins" if d_mae > 0 else "✗ AdaDelta wins"))
            print(f"    ΔRMSE = {d_rmse:+.4f}  "
                  + ("✓ LW wins" if d_rmse > 0 else "✗ AdaDelta wins"))

            lw_m  = [r['test_mae'] for r in ds_res[pred_len]['LW-AdaDelta']]
            ad_m  = [r['test_mae'] for r in ds_res[pred_len]['AdaDelta']]
            if len(lw_m) >= 2:
                t, p = stats.ttest_rel(lw_m, ad_m)
                print(f"    t={t:.3f}  p={p:.4f}  "
                      + ("✓ p<0.05" if p < 0.05 else "✗ not significant"))

    return all_metrics


# ============================================================
# Visualization
# ============================================================

def plot_results(results, all_metrics):
    optimizers    = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    pred_horizons = sorted(list(next(iter(results.values())).keys()))
    ds_names      = list(results.keys())
    colors = {
        'SGD':         '#6b7280', 'Adam':        '#3b82f6',
        'AdamW':       '#f59e0b', 'Lion':        '#14b8a6',
        'AdaDelta':    '#ef4444', 'LW-AdaDelta': '#10b981',
    }

    fig, axes = plt.subplots(len(ds_names), 3,
                             figsize=(18, 6 * len(ds_names)))
    if len(ds_names) == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle('ETT Load Forecasting: LW-AdaDelta vs Baselines\n'
                 'Dilated TCN | OT feature | MAE metric',
                 fontsize=14, fontweight='bold')

    for row, ds in enumerate(ds_names):

        # Col 0: MAE vs horizon
        ax = axes[row, 0]
        for opt in optimizers:
            means = [all_metrics[ds][h][opt]['mae_mean'] for h in pred_horizons]
            stds  = [all_metrics[ds][h][opt]['mae_std']  for h in pred_horizons]
            lw = 2.5 if opt in ('LW-AdaDelta', 'AdaDelta') else 1.2
            al = 1.0 if opt in ('LW-AdaDelta', 'AdaDelta') else 0.6
            ax.plot(pred_horizons, means, 'o-', label=opt,
                    color=colors[opt], linewidth=lw, alpha=al, markersize=7)
            ax.fill_between(pred_horizons,
                            np.array(means)-np.array(stds),
                            np.array(means)+np.array(stds),
                            alpha=0.12, color=colors[opt])
        ax.set_xlabel('Forecast Horizon (hours)', fontsize=11)
        ax.set_ylabel('Test MAE (scaled)', fontsize=11)
        ax.set_title(f'{ds}: MAE vs Horizon', fontsize=12, fontweight='bold')
        ax.set_xticks(pred_horizons)
        ax.set_xticklabels(['24h\n(1-day)', '48h\n(2-day)', '168h\n(1-week)'])
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Col 1: ΔMAE bars
        ax = axes[row, 1]
        deltas = [
            all_metrics[ds][h]['AdaDelta']['mae_mean'] -
            all_metrics[ds][h]['LW-AdaDelta']['mae_mean']
            for h in pred_horizons
        ]
        bar_c = ['#10b981' if d > 0 else '#ef4444' for d in deltas]
        bars  = ax.bar(['24h', '48h', '168h'], deltas, color=bar_c,
                       alpha=0.8, edgecolor='black', linewidth=1.5)
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        for bar, d in zip(bars, deltas):
            va  = 'bottom' if d >= 0 else 'top'
            off = max(abs(d)*0.05, 0.0002) * (1 if d >= 0 else -1)
            ax.text(bar.get_x()+bar.get_width()/2, d+off,
                    f'{d:+.4f}', ha='center', va=va,
                    fontsize=10, fontweight='bold')
        ax.set_ylabel('ΔMAE (AdaDelta − LW-AdaDelta)', fontsize=11)
        ax.set_title(f'{ds}: Improvement over AdaDelta\n'
                     '(green = LW-AdaDelta wins)',
                     fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')

        # Col 2: Val MAE curve at H=168, median seed
        ax       = axes[row, 2]
        med_seed = len(list(results[ds][168]['Adam'])) // 2
        for opt in optimizers:
            curve = results[ds][168][opt][med_seed]['history']['val_mae']
            lw = 2.5 if opt in ('LW-AdaDelta', 'AdaDelta') else 1.2
            al = 1.0 if opt in ('LW-AdaDelta', 'AdaDelta') else 0.55
            ax.plot(curve, label=opt, color=colors[opt],
                    linewidth=lw, alpha=al)
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('Validation MAE', fontsize=11)
        ax.set_title(f'{ds}: Val MAE Curve (H=168h, median seed)',
                     fontsize=12, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out = '/kaggle/working/ett_forecasting_results.png'
    plt.savefig(out, dpi=300, bbox_inches='tight')
    print(f"\n✓ Plot saved: {out}")
    plt.close()


# ============================================================
# Paper Summary Table
# ============================================================

def print_summary_table(all_metrics):
    optimizers    = ['Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    pred_horizons = [24, 48, 168]

    print("\n" + "="*80)
    print("PAPER-READY SUMMARY TABLE (MAE ± std)")
    print("="*80)

    header = f"{'Dataset':<10} {'H':<8} " + \
             "  ".join(f"{o:<22}" for o in optimizers)
    print(header)
    print("-" * len(header))

    for ds in all_metrics:
        for h in pred_horizons:
            best = min(all_metrics[ds][h][o]['mae_mean'] for o in optimizers)
            row  = f"{ds:<10} {str(h)+'h':<8} "
            for o in optimizers:
                m   = all_metrics[ds][h][o]
                val = f"{m['mae_mean']:.4f}±{m['mae_std']:.4f}"
                if abs(m['mae_mean'] - best) < 1e-6:
                    val = f"*{val}*"
                row += f"{val:<24}"
            print(row)

    print("\n* = best MAE per row")
    print("\nKey claim: LW-AdaDelta advantage should grow H=24 → H=48 → H=168")
    print("  Longer horizon = more temporal stress = bigger LW advantage")


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device : {device}")
    print(f"GPUs available: {torch.cuda.device_count()}")

    if torch.cuda.device_count() > 1:
        print(f"✓ DataParallel will activate for all experiments")

    # Step 1 — Download (tiny, ~5 seconds)
    DATA_DIR  = './ett_data'
    ett_paths = download_ett(DATA_DIR)

    # Step 2 — Run benchmark
    print("\n" + "="*80)
    print("ETT Load Forecasting — Full Benchmark")
    print("="*80)
    print("Datasets      : ETTh1, ETTh2")
    print("Horizons      : 24h, 48h, 168h")
    print("Optimizers    : 6")
    print("Seeds         : 3")
    print("Total runs    : 108")
    print("Batch size    : 64  (up from 32)")
    print("num_workers   : 2")
    print("DataParallel  : enabled")
    print("Checkpoint    : after every run")
    print("Est. runtime  : ~2 hours on T4x2")
    print("="*80)

    results = run_full_benchmark(ett_paths, device=device, seeds=[0, 1, 2])

    # Step 3 — Analyze
    all_metrics = analyze_results(results)

    # Step 4 — Plot
    plot_results(results, all_metrics)

    # Step 5 — Paper table
    print_summary_table(all_metrics)

    # Step 6 — Save finals
    with open('/kaggle/working/ett_final_results.pkl', 'wb') as f:
        pickle.dump(results, f)

    print("\n" + "="*80)
    print("BENCHMARK COMPLETE!")
    print("="*80)
    print("✓ Results : /kaggle/working/ett_final_results.pkl")
    print("✓ Plot    : /kaggle/working/ett_forecasting_results.png")
    print("\nKey claims to verify:")
    print("  • ΔMAE grows from H=24 → H=168 (growing advantage at longer horizons)")
    print("  • LW-AdaDelta competitive with Adam/AdamW across all conditions")
    print("  • Val MAE curve: LW-AdaDelta separates from AdaDelta at H=168")
    print("\n>>> STOP THE SESSION NOW: Run menu → Stop Session <<<")

Using device : cuda
GPUs available: 2
✓ DataParallel will activate for all experiments
[INFO] Downloading ETTh1 …
[INFO] Saved → ./ett_data/ETTh1.csv
[INFO] Downloading ETTh2 …
[INFO] Saved → ./ett_data/ETTh2.csv

ETT Load Forecasting — Full Benchmark
Datasets      : ETTh1, ETTh2
Horizons      : 24h, 48h, 168h
Optimizers    : 6
Seeds         : 3
Total runs    : 108
Batch size    : 64  (up from 32)
num_workers   : 2
DataParallel  : enabled
Checkpoint    : after every run
Est. runtime  : ~2 hours on T4x2
[INFO] No checkpoint — starting fresh
[INFO] Loaded ETTh1.csv: shape=(17420, 7), cols=['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
[INFO] Train=12194, Val=1742, Test=3484 timesteps

######################################################################
# Dataset=ETTh1  Horizon=24h  Seq=336  Features=7
######################################################################

[Progress 1/108] ETTh1 H=24 SGD seed=0
[INFO] Loader sizes — Train:11835 Val:1383 Test:3125
  Epoch   1: Lo